# FinSight AI — Research Notebook 01: Exploratory Financial Data Analysis (EDA)
## B.Tech Major Project in Computer Science & Artificial Intelligence

This notebook analyzes daily OHLCV equity series, returns distributions, skewness, kurtosis, and rolling volatility regimes across major securities (AAPL, MSFT, NVDA, SPY).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "backend"))  # run from the ml_research/ folder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, jarque_bera
from app.data import market_data as md

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# 1. Daily closes (live Yahoo Finance data, simulated fallback when offline)
tickers = ["AAPL", "MSFT", "NVDA", "SPY"]
prices, simulated = md.closes(tickers, period="5y")
print("Simulated series:", simulated or "none")
prices.tail()

In [ ]:
# 2. Daily log returns
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.describe().T

In [ ]:
# 3. Fat tails & non-normality: skewness, excess kurtosis, Jarque-Bera test
rows = []
for t in log_returns.columns:
    r = log_returns[t]
    jb_stat, jb_p = jarque_bera(r)
    rows.append({"ticker": t, "skew": skew(r), "excess_kurtosis": kurtosis(r), "jarque_bera_p": jb_p,
                 "annual_vol": r.std() * np.sqrt(252)})
pd.DataFrame(rows).set_index("ticker").round(4)

In [ ]:
# 4. Growth of $1 — cumulative return from log returns is exp(cumsum)
growth = np.exp(log_returns.cumsum())
growth.plot(title="Growth of $1 over 5 years")
plt.ylabel("Value of $1")
plt.show()

In [ ]:
# 5. Rolling 30-day annualised volatility — volatility clusters in regimes
(log_returns.rolling(30).std() * np.sqrt(252)).plot(title="Rolling 30-day volatility (annualised)")
plt.show()